In [1]:
import pandas as pd
import numpy as np

In [17]:

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df_main = pd.read_csv(url)
df_main.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [18]:
df_main.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [19]:
categorical = list(df_main.select_dtypes(include=["object", "category", "string"]).columns)
numerical = list(df_main.select_dtypes(include="number").columns)
del numerical[-1]

# Handle missing values
df_main[categorical] = df_main[categorical].fillna("NA")
df_main[numerical] = df_main[numerical].fillna(0)


In [190]:
df_main.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [191]:
df_main['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [192]:
df_main[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [193]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df_main, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train["converted"]
y_test = df_test["converted"]
y_val = df_val["converted"]

del df_train["converted"]
del df_test["converted"]
del df_val["converted"]

df_train.reset_index(inplace=True, drop=True)
df_test.reset_index(inplace=True, drop=True)
df_val.reset_index(inplace=True, drop=True)



In [194]:
from sklearn.metrics import mutual_info_score
for c in categorical:
    score = mutual_info_score(df_train[c], y_train)
    print(c, score)

lead_source 0.03434605791617973
industry 0.0019945636715935217
employment_status 0.018965604906982875
location 0.001042763579701525


In [203]:
from sklearn.feature_extraction import DictVectorizer

dict_train = df_train[categorical + numerical].to_dict(orient='records')
dict_test = df_test[categorical + numerical].to_dict(orient='records')
dict_val = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(dict_train)
X_test = dv.transform(dict_test)
X_val = dv.transform(dict_val)

dv.feature_names_

['annual_income',
 'employment_status=NA',
 'employment_status=employed',
 'employment_status=self_employed',
 'employment_status=student',
 'employment_status=unemployed',
 'industry=NA',
 'industry=education',
 'industry=finance',
 'industry=healthcare',
 'industry=manufacturing',
 'industry=retail',
 'industry=technology',
 'interaction_count',
 'lead_score',
 'lead_source=NA',
 'lead_source=events',
 'lead_source=organic_search',
 'lead_source=paid_ads',
 'lead_source=referral',
 'lead_source=social_media',
 'location=NA',
 'location=africa',
 'location=asia',
 'location=europe',
 'location=north_america',
 'location=south_america',
 'number_of_courses_viewed']

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [199]:
y_pred = model.predict_proba(X_val)[:,-1]
decision = (y_pred >= 0.5).astype(int)
accuracy_original = np.round((decision == y_val).mean(), 4)
accuracy_original

0.645

In [202]:
features = categorical + numerical
features

['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [ ]:
features = categorical + numerical
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    features_new = features.copy()
    df_small_train = df_train.copy()
    df_small_val = df_val.copy()

    del df_small_train[feature]
    del df_small_val[feature]

    features_new.remove(feature)

    dict_train_small = df_small_train[features_new].to_dict(orient='records')
    dict_val_small = df_small_val[features_new].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)

    X_train_new = dv.fit_transform(dict_train_small)
    X_val_new = dv.transform(dict_val_small)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_new, y_train)

    y_pred = model.predict_proba(X_val_new)[:,-1]
    decision = (y_pred >= 0.5).astype(int)
    accuracy = np.round((decision == y_val).mean(), 2)
    print(feature, accuracy_original - accuracy)



lead_source 0.0050000000000000044
number_of_courses_viewed 0.0050000000000000044
interaction_count 0.04500000000000004


In [ ]:
for c in [0.000001, 0.00001, 0.0001, 0.001]: 
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:,-1]
    decision = (y_pred >= 0.5).astype(int)
    accuracy = np.round((decision == y_val).mean(), 4)
    print(c, accuracy)
    

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
